# SSI FastConnect — kiểm tra REST + luồng tool của TradingAgents

Notebook này kiểm tra theo thứ tự: (1) kết nối SSI chỉ đọc, (2) lựa chọn nguồn dữ liệu trong router, (3) tool giá OHLCV và tool RSI mà các agent thực sự gọi. Không gọi LLM, không đặt lệnh.

**Credentials:** đọc từ biến môi trường runtime trước, rồi Colab Secrets; chỉ hỏi nhập ẩn nếu chưa có. Chúng chỉ ở bộ nhớ runtime, không được in ra hoặc lưu vào notebook/GitHub. Muốn xóa khỏi phiên, chọn Runtime → Disconnect and delete runtime.

In [ ]:
import os, subprocess, sys
from pathlib import Path

REPO_URL = "https://github.com/PhamHuuQan/TradingAgents.git"
BRANCH = "feature/vietnam-market-foundation"
REPO_DIR = Path("/content/TradingAgents")

if not REPO_DIR.exists():
    subprocess.run(["git", "clone", "--depth", "1", "--branch", BRANCH, REPO_URL, str(REPO_DIR)], check=True)
else:
    subprocess.run(["git", "-C", str(REPO_DIR), "fetch", "--depth", "1", "origin", BRANCH], check=True)
    subprocess.run(["git", "-C", str(REPO_DIR), "checkout", "-B", BRANCH, "FETCH_HEAD"], check=True)

# Install the project dependencies so this checks the actual agent tools/router,
# not only the standalone SSI connector.
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", ".[dev]"], cwd=str(REPO_DIR), check=True)
sys.path.insert(0, str(REPO_DIR))
print("TradingAgents branch checked out and project dependencies installed.")

In [ ]:
from datetime import datetime, timedelta
from getpass import getpass
from zoneinfo import ZoneInfo
import os

def read_colab_secret(name):
    # Reuse the current runtime first, then Colab Secrets.
    value = os.environ.get(name)
    if value and value.strip():
        return value.strip()
    try:
        from google.colab import userdata
        value = userdata.get(name)
    except Exception:
        value = None
    return value.strip() if isinstance(value, str) and value.strip() else None

api_key = read_colab_secret("SSI_API_KEY") or getpass("SSI_API_KEY (input hidden): ")
api_secret = read_colab_secret("SSI_API_SECRET") or getpass("SSI_API_SECRET (input hidden): ")
if not api_key.strip() or not api_secret.strip():
    raise ValueError("Both SSI credentials are required; no credentials were saved to the notebook.")

os.environ["SSI_API_KEY"] = api_key.strip()
os.environ["SSI_API_SECRET"] = api_secret.strip()
del api_key, api_secret

today_vn = datetime.now(ZoneInfo("Asia/Ho_Chi_Minh")).date()
from_date = (today_vn - timedelta(days=30)).isoformat()
to_date = today_vn.isoformat()

print(f"[1/3] Direct SSI REST smoke test: FPT, {from_date} to {to_date}.")
from tradingagents.dataflows.vietnam_market.smoke import main as smoke_main
exit_code = smoke_main(["--symbol", "FPT", "--from-date", from_date, "--to-date", to_date, "--page-size", "100"])
if exit_code != 0:
    raise RuntimeError("SSI REST smoke test failed. Read the sanitized error above; do not share credentials.")

print("\n[2/3] Verify that TradingAgents selected SSI for both prices and technical indicators.")
from tradingagents.dataflows.config import set_config
from tradingagents.dataflows.router import get_vendor
# Explicitly set the run's data source even if modules were imported earlier.
set_config({"data_vendors": {"core_stock_apis": "ssi", "technical_indicators": "ssi"}})
selected_prices = get_vendor("core_stock_apis", "get_stock_data")
selected_indicators = get_vendor("technical_indicators", "get_indicators")
print(f"Price-data vendor: {selected_prices}")
print(f"Indicator vendor: {selected_indicators}")
if selected_prices.split(",")[0].strip() != "ssi" or selected_indicators.split(",")[0].strip() != "ssi":
    raise RuntimeError("TradingAgents did not select SSI as the primary vendor; refusing to claim integration success.")

print("\n[3/3] Call the actual agent tools (no LLM/API bill; read-only market data only).")
from tradingagents.agents.tools import get_stock_data, get_indicators
price_result = get_stock_data.func("FPT", from_date, to_date, to_date)
if not isinstance(price_result, str) or "# SSI FastConnect V3 read-only OHLCV for FPT" not in price_result:
    raise RuntimeError("Agent get_stock_data tool did not return an SSI-sourced response.")
price_rows = [
    line for line in price_result.splitlines()
    if line.strip() and not line.startswith("#") and not line.startswith("Date,")
]
if not price_rows:
    raise RuntimeError("Agent get_stock_data returned no OHLCV rows.")
print(f"PASS — agent get_stock_data uses SSI and returned {len(price_rows)} OHLCV rows.")

indicator_result = get_indicators.func("FPT", "rsi", to_date, 30, to_date)
if not isinstance(indicator_result, str) or "# SSI FastConnect V3 indicator rsi for FPT" not in indicator_result:
    raise RuntimeError("Agent get_indicators tool did not return an SSI-sourced indicator.")
indicator_rows = [
    line for line in indicator_result.splitlines()
    if line.strip() and not line.startswith("#") and not line.startswith("Date,")
]
if not indicator_rows:
    raise RuntimeError("Agent get_indicators returned no RSI values.")
print(f"PASS — agent get_indicators uses SSI and returned {len(indicator_rows)} RSI points.")
print("\n✅ COMPLETE: SSI REST, data routing, and both agent tools passed read-only checks.")

# Credentials remain only in this Colab runtime's memory; they are never printed or saved to disk.
# Disconnect/delete the runtime to clear them.


## Bước tiếp theo: chạy một lượt phân tích AI thật

Ô kế tiếp chạy một lượt `TradingAgentsGraph` cho FPT bằng Gemini và đặt SSI làm nguồn giá/chỉ báo. Để giảm số lượt gọi ban đầu, chỉ chọn Market Analyst; phần researcher, trader và risk/portfolio trong graph vẫn chạy. Lượt này có thể tạo nhiều request Gemini và chịu hạn mức dự án. Trên Free Tier, Gemini 3.8 Flash hiện có token miễn phí nhưng giới hạn RPM/TPM/RPD; dữ liệu gửi lên Free Tier có thể được dùng để cải thiện sản phẩm của Google. Không chạy lại nhiều lần liên tiếp nếu không cần.

In [ ]:
from datetime import datetime
from copy import deepcopy
from pathlib import Path
from zipfile import ZipFile, ZIP_DEFLATED
from zoneinfo import ZoneInfo
from getpass import getpass
import os

MODEL_ID = "gemini-3.8-flash"  # Stable model ID in Google's current Gemini API model list.

# Read GOOGLE_API_KEY from the runtime or Colab Secrets; prompt hidden only if absent.
google_key = read_colab_secret("GOOGLE_API_KEY") or getpass("GOOGLE_API_KEY (input hidden): ")
if not google_key.strip():
    raise ValueError("GOOGLE_API_KEY is empty.")
os.environ["GOOGLE_API_KEY"] = google_key.strip()
del google_key

# Verify the key/model before starting the multi-agent graph, so a bad key
# doesn't waste many requests. Never print the key or the full raw exception.
from langchain_google_genai import ChatGoogleGenerativeAI
try:
    ping = ChatGoogleGenerativeAI(
        model=MODEL_ID,
        google_api_key=os.environ["GOOGLE_API_KEY"],
        max_retries=0,
        max_output_tokens=32,
        timeout=60,
    ).invoke("Reply with exactly the single word READY.")
    if not getattr(ping, "content", None):
        raise RuntimeError("Gemini returned an empty response.")
    print(f"✅ Gemini API preflight passed: {MODEL_ID}")
except Exception as exc:
    message = str(exc).replace(os.environ["GOOGLE_API_KEY"], "[REDACTED]")
    raise RuntimeError(
        f"Gemini preflight failed ({type(exc).__name__}): {message[:700]}"
    ) from None

from tradingagents.default_config import DEFAULT_CONFIG
from tradingagents.graph.trading_graph import TradingAgentsGraph

# Explicit run config: Gemini for both tiers, SSI for Vietnamese equity price/indicators,
# one market analyst for the first end-to-end test, and bounded debate/tool rounds.
config = deepcopy(DEFAULT_CONFIG)
config["llm_provider"] = "google"
config["quick_think_provider"] = None
config["deep_think_provider"] = None
config["quick_think_llm"] = MODEL_ID
config["deep_think_llm"] = MODEL_ID
config["output_language"] = "Vietnamese"
config["max_debate_rounds"] = 1
config["max_risk_discuss_rounds"] = 1
config["max_tool_rounds"] = 4
config["max_recur_limit"] = 100
config["max_tokens"] = 2048
config["llm_max_retries"] = 1
config["data_vendors"] = dict(config.get("data_vendors", {}))
config["data_vendors"]["core_stock_apis"] = "ssi"
config["data_vendors"]["technical_indicators"] = "ssi"

# This adapter blocks backdated SSI data because historical first-availability
# timestamps are not known. Use the current Vietnam market date for this live test.
analysis_date = datetime.now(ZoneInfo("Asia/Ho_Chi_Minh")).date().isoformat()
print(f"Starting TradingAgents end-to-end test: FPT, {analysis_date}")
print("Selected analyst: market. Graph still runs research, trader, risk and portfolio stages.")
print("Data sources: SSI prices/indicators; other categories retain the repository config.")

ta = TradingAgentsGraph(
    selected_analysts=["market"],
    config=config,
    debug=False,
)
final_state, signal = ta.propagate("FPT", analysis_date, asset_type="stock")
report_dir = Path(ta.save_reports(final_state, "FPT", html=True))

decision = final_state.get("final_trade_decision", "")
print("\n✅ TradingAgents graph completed.")
print(f"Signal/rating: {signal}")
print(f"Final trade decision: {str(decision)[:2500]}")
print(f"Report folder: {report_dir}")

# Package the report tree so it can be downloaded from Colab.
archive = Path("/content/TradingAgents_FPT_report.zip")
with ZipFile(archive, "w", compression=ZIP_DEFLATED) as zf:
    for file in report_dir.rglob("*"):
        if file.is_file():
            zf.write(file, arcname=str(Path(report_dir.name) / file.relative_to(report_dir)))

from IPython.display import FileLink, display
print("\nDownload the generated report bundle:")
display(FileLink(str(archive)))
